# Cost per valid failure

Uses the shared result table and simple editable assumptions. This is not GPU benchmarking; it is an engineering-cost sensitivity view.

In [ ]:
from pathlib import Path
import csv, math, subprocess

REPO = 'https://github.com/casparbreloh/rbt4dnn-seminar.git'
candidates = [Path.cwd()]

try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_base = Path('/content/drive/MyDrive/Studium/Seminar - RBT4DNN')
    candidates += [drive_base / 'experiments', drive_base / 'Experiments']
except Exception:
    pass

if not any((p / 'results.csv').exists() for p in candidates):
    target = Path('/content/rbt4dnn-seminar')
    if not target.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO, str(target)], check=True)
    candidates.append(target)

ROOT = next((p for p in candidates if (p / 'results.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Could not find results.csv')

with open(ROOT / 'results.csv', newline='') as f:
    rows = list(csv.DictReader(f))

print(ROOT)


In [1]:
ASSUMPTIONS = {
    'gpu_hour_cost': 3.00,
    'per_requirement_train_hours': 1.00,
    'shared_train_hours': 1.50,
    'generation_hours_per_1000': 0.10,
    'human_validation_cost_per_image': 0.03,
    'engineer_hours_per_requirement': 2.00,
    'engineer_hourly_rate': 25.00,
}

def f(x):
    return None if x == '' else float(x)

def cost(r):
    n = int(r['n_images'])
    if r['variant'] == 'lr':
        train = ASSUMPTIONS['per_requirement_train_hours']
        eng = ASSUMPTIONS['engineer_hours_per_requirement']
    else:
        train = ASSUMPTIONS['shared_train_hours'] / 6
        eng = ASSUMPTIONS['engineer_hours_per_requirement'] / 2
    return (
        ASSUMPTIONS['gpu_hour_cost'] * (train + ASSUMPTIONS['generation_hours_per_1000'] * n / 1000)
        + ASSUMPTIONS['human_validation_cost_per_image'] * n
        + eng * ASSUMPTIONS['engineer_hourly_rate']
    )

print('dataset req variant | pass | match | valid failures | cost/failure')
for r in rows:
    p, m = f(r['pass_rate_mean']), f(r['precondition_match_mean'])
    if p is None or m is None or not r['n_images']:
        continue
    valid_failures = int(r['n_images']) * m * (1 - p)
    if valid_failures <= 0:
        continue
    if r['dataset'] == 'mnist' and r['requirement'] == 'M3' or r['dataset'] in ['celeba', 'sgsm'] and valid_failures > 100:
        print(f"{r['dataset']} {r['requirement']} {r['variant']} | {p:.3f} | {m:.3f} | {valid_failures:.1f} | ${cost(r)/valid_failures:.2f}")


dataset req variant | pass | match | valid failures | cost/failure
mnist M3 lr | 0.694 | 0.958 | 2930.5 | $0.12
celeba C1 lr | 0.916 | 0.961 | 809.2 | $0.44
celeba C2 lr | 0.879 | 0.964 | 1170.3 | $0.30
celeba C3 lr | 0.914 | 0.975 | 837.5 | $0.43
celeba C4 lr | 0.909 | 0.894 | 813.5 | $0.44
celeba C5 lr | 0.975 | 0.955 | 240.7 | $1.48
celeba C6 lr | 0.796 | 0.981 | 2004.2 | $0.18
sgsm S1 lr | 0.653 | 0.760 | 2633.4 | $0.14
sgsm S2 lr | 0.134 | 0.850 | 7364.4 | $0.05
sgsm S3 lr | 0.887 | 0.783 | 884.8 | $0.40
mnist M3 allreq | 0.932 | 0.275 | 18.7 | $3.00
mnist M3 alldata | 0.996 | 0.074 | 0.3 | $189.36


Takeaway: the metric becomes meaningful only after accounting for precondition match. For MNIST, the all-data LoRA looks good by raw pass rate but produces almost no valid M3 failures.